### multi head attention

In [22]:
import numpy as np
import torch.nn as nn
import math
import torch

In [31]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim_in, dim_out, context_length, number_of_heads, dropout_percenatage=0.0, qkv_bias=False):
        super().__init__()
        
        self.dim_in = dim_in
        self.dim_out = dim_out
        self.context_length = context_length
        self.number_of_heads = number_of_heads

        self.dropout = nn.Dropout(p=dropout_percenatage)
        
        self.dimension_per_head = dim_out // number_of_heads

        self.weight_query = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.weight_key = nn.Linear(dim_in, dim_out, bias=qkv_bias)
        self.weight_value = nn.Linear(dim_in, dim_out, bias=qkv_bias)

        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, batch):
        batch_size, number_of_tokens, dim_in = batch.shape

        queries = self.weight_query(batch)
        keys = self.weight_key(batch)
        values = self.weight_value(batch)

        keys = keys.view(batch_size, number_of_tokens, self.number_of_heads, self.dimension_per_head)
        queries = queries.view(batch_size, number_of_tokens, self.number_of_heads, self.dimension_per_head)
        values = values.view(batch_size, number_of_tokens, self.number_of_heads, self.dimension_per_head)

        # Before the transpose, your data is organized by Token:Shape: (Batch, Tokens, Heads, Depth)Meaning: "For this sentence, look at the 1st word, then its 1st head."After the transpose(1, 2), the data is organized by Head:Shape: (Batch, Heads, Tokens, Depth)Meaning: "For this sentence, look at the 1st head, then all its words." 

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attention_scores = queries @ keys.transpose(2,3)

        causal_mask = self.mask.bool()

        causal_attention_scores = attention_scores.masked_fill(causal_mask, -torch.inf)

        causal_scaled_weights = causal_attention_scores / math.sqrt(keys.shape[-1])

        causal_attention_weights = torch.softmax(causal_scaled_weights, dim=-1)

        causal_attention_weights = self.dropout(causal_attention_weights)

        context_vector = (causal_attention_weights @ values).transpose(1,2)

        context_vector = context_vector.contiguous().view(batch_size, number_of_tokens, self.dim_out)

        return context_vector

In [32]:
# input embeddings
inputs = torch.tensor([
   [0.43, 0.15, 0.89], # Your 
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]  # step
])

In [33]:
batch = torch.stack((inputs, inputs), dim=0)
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [34]:
batch.shape

torch.Size([2, 6, 3])

In [36]:
dim_in = inputs.shape[-1]
dim_out = 2
context_length = inputs.shape[0]

multi_head_attention = MultiHeadAttention(dim_in=dim_in, dim_out=dim_out, context_length=context_length,number_of_heads=2)
multi_head_attention
context_vector = multi_head_attention(batch)
context_vector


tensor([[[0.1297, 0.0007],
         [0.0616, 0.0980],
         [0.0416, 0.1322],
         [0.0219, 0.1281],
         [0.0375, 0.1431],
         [0.0189, 0.1350]],

        [[0.1297, 0.0007],
         [0.0616, 0.0980],
         [0.0416, 0.1322],
         [0.0219, 0.1281],
         [0.0375, 0.1431],
         [0.0189, 0.1350]]], grad_fn=<ViewBackward0>)